# 03c — Two-Stage Grid · CLF (Stage 1)

**역할**: 분류기 1개를 학습 → die-level prob csv 저장. 03e에서 이 prob을 03d reg와 곱.

- 전처리 = `final/modules/preprocess.run` DEFAULT_PARAMS (1차 baseline best 박힘)
- 학습 = die-level binary, KFold(unit 단위) OOF
- objective = `RMSE(unit_mean(die_proba) × y_pos_const, y_train_unit)` ← clf calibration RMSE 단독 평가
- `CLF_MODEL_NAME` 스위치로 한 번 실행 = 한 모델. **코랩 병렬**로 모델별 동시 실행.

출력: `4_output/final/two_stage_grid/clf/{CLF_MODEL_NAME}/{oof,val,test}_die.csv` + `_unit.csv` + `fold_models.pkl` + `best_params.json`

## 1. 환경 설정 + 모듈 import (Colab/Local 공통)

In [1]:
import os, sys

# ── Colab 사용 시에만 채울 것 (로컬은 무시) ──
GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'   # ★ Colab에서 final.zip 업로드 후 공유 ID 입력

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/hpo.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID가 비어있음 — final.zip 공유 ID를 입력하세요'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess, hpo, models

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available CLF models: {models.CLF_AVAILABLE_MODELS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
Available CLF models: ['lgbm', 'xgb', 'catboost', 'et']


## 2. 실험 설정

- `CLF_MODEL_NAME` 스위치: 한 번 실행 = 한 모델 학습. 코랩 병렬은 모델당 노트북 1개씩 잡고 동시 실행.
- `PARAMS = {}` 빈 dict로 호출 → `final/modules/preprocess.py` `DEFAULT_PARAMS` (= baseline 1차 best) 그대로.

In [ ]:
# ── 모델 선택 (한 번에 1개) ──
CLF_MODEL_NAME = 'lgbm'        # ★ 'lgbm' | 'xgb' | 'catboost' | 'et'
assert CLF_MODEL_NAME in models.CLF_AVAILABLE_MODELS, \
    f'CLF_MODEL_NAME invalid: {CLF_MODEL_NAME}'

# ── 실험 식별 ──
EXP_ID   = f'ts-clf-{CLF_MODEL_NAME}-001'
EXP_MEMO = f'Two-Stage Grid · CLF · {CLF_MODEL_NAME}'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS = 1
N_FOLDS  = 5

# ── target clip (binary label에는 무영향, 일관성 위해 유지) ──
CLIP_Y_EXTREME = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'two_stage_grid', 'clf', CLF_MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 PARAMS — 빈 dict면 final/modules/preprocess.py DEFAULT (baseline 1차 best) ──
PARAMS = {}

# ── 디바이스 (필요시 'gpu') ──
# models.DEVICE = 'gpu'

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'CLF_MODEL_NAME: {CLF_MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}')
print(f'CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DEVICE={models.DEVICE}')
print(f'PARAMS override: {PARAMS}  (빈 dict → preprocess.py DEFAULT_PARAMS 그대로)')

## 3. 데이터 로드 + Y clip + 전처리

In [3]:
# ── 데이터 로드 ──
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# ── y_train 극단값 clip (binary label에는 무영향이지만 일관성 위해) ──
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# ── 전처리 (DEFAULT_PARAMS, 빈 override) ──
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  xs_train: {xs_train.shape}, val: {xs_val.shape}, test: {xs_test.shape}')

# ── binary 분포 확인 ──
y_train_unit = ys_input['train']
n_pos = (y_train_unit[TARGET_COL] > 0).sum()
n_neg = (y_train_unit[TARGET_COL] == 0).sum()
print(f'\nUnit binary 분포: pos={n_pos:,} ({n_pos/(n_pos+n_neg):.1%}), neg={n_neg:,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,7

## 4. Optuna HPO (CLF)

- `run_clf_hpo`: KFold(unit 단위) OOF
- objective = `RMSE(unit_mean(die_proba) × y_pos_const, y_train_unit)` (clf 단독 RMSE 평가)
- imbalance 자동 처리: `scale_pos_weight` (lgbm/xgb), `auto_class_weights='Balanced'` (catboost), `class_weight='balanced'` (et) — fold-local

In [4]:
study_meta = {
    'exp_id':           EXP_ID,
    'exp_memo':         EXP_MEMO,
    'user':             USER,
    'clf_model_name':   CLF_MODEL_NAME,
    'clip_y_extreme':   CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':         N_TRIALS,
    'n_folds':          N_FOLDS,
    'seed':             SEED,
}

res = hpo.run_clf_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=CLF_MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    user_attrs=study_meta,
    # ★ trial별 holdout RMSE 기록 (DB trial_user_attributes)
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study       = res['study']
best_params = res['best_params']
y_pos_const = res['y_pos_const']

print(f'\n[HPO 완료] best train RMSE = {res["best_value"]:.6f}')
print(f'y_pos_const (E[Y|Y>0]) = {y_pos_const:.6f}')
print(f'best_params = {best_params}')

[I 2026-05-02 16:25:54,160] A new study created in RDB with name: ts-clf-lgbm-001


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-05-02 16:27:30,876] Trial 0 finished with value: 0.00565492054508719 and parameters: {'n_estimators': 1186, 'learning_rate': 0.24517932047070642, 'num_leaves': 283, 'max_depth': 10, 'min_child_samples': 66, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.15227525095137953, 'reg_alpha': 1.6175449623854197, 'reg_lambda': 0.002570603566117598, 'min_split_gain': 0.0023585940584142655, 'path_smooth': 1.0292247147901223}. Best is trial 0 with value: 0.00565492054508719.

[HPO 완료] best train RMSE = 0.005655
y_pos_const (E[Y|Y>0]) = 0.008496
best_params = {'n_estimators': 1186, 'learning_rate': 0.24517932047070642, 'num_leaves': 283, 'max_depth': 10, 'min_child_samples': 66, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.15227525095137953, 'reg_alpha': 1.6175449623854197, 'reg_lambda': 0.002570603566117598, 'min_split_gain': 0.0023585940584142655, 'path_smooth': 1.0292247147901223}


## 5. Best trial 재학습 (K-fold OOF) + die-level prob 캡쳐

In [5]:
final = hpo.refit_clf_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=CLF_MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
)

# ── unit-level RMSE 계산 (clf 단독 평가) ──
def _unit_mean_proba(xs_split, die_proba):
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'p': die_proba})
    return df.groupby(KEY_COL, sort=False)['p'].mean()

def _rmse(pred_unit, y_unit_df):
    aligned = pred_unit.loc[y_unit_df.set_index(KEY_COL).index]
    return float(np.sqrt(np.mean((aligned.values - y_unit_df[TARGET_COL].values) ** 2)))

oof_unit_proba  = _unit_mean_proba(xs_train, final['oof_proba_die'])
val_unit_proba  = _unit_mean_proba(xs_val,   final['val_proba_die'])
test_unit_proba = _unit_mean_proba(xs_test,  final['test_proba_die'])

oof_rmse  = _rmse(oof_unit_proba * y_pos_const, ys_input['train'])
val_rmse  = _rmse(val_unit_proba * y_pos_const, ys_input['validation'])
test_rmse = _rmse(test_unit_proba * y_pos_const, ys_input['test'])

print(f'\n[Refit 완료] (clf 단독, prob × y_pos_const)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')
print(f'fold_models: {len(final["fold_models"])}개')

[clf refit fold 1/5] tr_units=20949, vl_units=5238, pos_ratio=0.290
[clf refit fold 2/5] tr_units=20949, vl_units=5238, pos_ratio=0.294
[clf refit fold 3/5] tr_units=20950, vl_units=5237, pos_ratio=0.293
[clf refit fold 4/5] tr_units=20950, vl_units=5237, pos_ratio=0.292
[clf refit fold 5/5] tr_units=20950, vl_units=5237, pos_ratio=0.291

[Refit 완료] (clf 단독, prob × y_pos_const)
  OOF  unit RMSE = 0.005655
  val  unit RMSE = 0.005787
  test unit RMSE = 0.008462
fold_models: 5개


## 6. 산출물 저장 (`4_output/final/two_stage_grid/clf/{MODEL_NAME}/`)

In [6]:
hpo.save_clf_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    y_pos_const=y_pos_const,
    study_meta=study_meta,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

# ── Colab → 로컬 자동 다운로드 (로컬은 자동 skip) ──
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'clf_{CLF_MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
        print(f'[브라우저 다운로드 트리거] {os.path.basename(_zip_path)}')
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭해 수동 다운로드')
        display(FileLink(_zip_path))
except ImportError:
    pass

[save_clf_artifacts] c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_grid\clf\lgbm 저장 완료 (fold_models.pkl + best_params.json + 6 CSV)
  best_params.json                       8.9 KB
  fold_models.pkl                   96,595.5 KB
  oof_die.csv                        5,419.7 KB
  oof_unit.csv                       1,479.9 KB
  optuna_jh_ts-clf-lgbm-001.db         112.0 KB
  test_die.csv                       1,799.5 KB
  test_unit.csv                        492.4 KB
  val_die.csv                        1,799.4 KB
  val_unit.csv                         492.3 KB


## 7. 요약

In [7]:
print('=' * 75)
print(f'  CLF · {CLF_MODEL_NAME} ({EXP_ID})')
print('=' * 75)
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print(f'  trials            : {len(study.trials)}')
print(f'  y_pos_const       : {y_pos_const:.6f}  (E[Y|Y>0])')
print('-' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"unit RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  → die-level prob csv 저장 완료. 03e_ts_combine.ipynb 에서 reg와 곱.')
print(f'  → 다른 CLF 모델 돌리려면 CLF_MODEL_NAME 바꿔서 재실행.')
print('=' * 75)

  CLF · lgbm (ts-clf-lgbm-001)
  feat cols (clean) : 573
  trials            : 1
  y_pos_const       : 0.008496  (E[Y|Y>0])
---------------------------------------------------------------------------
                        OOF          val         test
  unit RMSE        0.005655     0.005787     0.008462
---------------------------------------------------------------------------
  → die-level prob csv 저장 완료. 03e_ts_combine.ipynb 에서 reg와 곱.
  → 다른 CLF 모델 돌리려면 CLF_MODEL_NAME 바꿔서 재실행.
